# HourProof carbon kill test: does AI make the buyer's hourly carbon footprint smaller, provably?

The idea: Denmark's IT and data-centre sector (the HourProof buyer) shifts its **flexible** load into the hours when the
grid is cleanest, using a **learned day-ahead CO2-intensity forecast**, and proves its **hourly residual emissions**
in zero knowledge without revealing its load.

Splits: TRAIN Jan–Mar 2025, VALIDATION Apr–Jun (the only place choices are made), TEST Jul–Dec (run once).
Real data only: Energinet (Energi Data Service). Runtime: Run all, CPU. First run ~15–40 min (downloads are cached).

| Test | PASS needs (set before running) |
|---|---|
| **K1** CO2 forecast | TEST error at least 30 % lower than the best persistence baseline |
| **K2** carbon shifting | Shifting 10 % flexible load cuts the buyer's location-based emissions by >= 1.0 %, costing <= 0.5 pp of hourly coverage |
| **K3** AI is load-bearing | The learned forecast reaches >= 50 % of the perfect-forecast reduction AND >= 1.25x what persistence achieves |
| **K4** ZK emissions proof | A real week: coverage and residual emissions proven in <= 60 s; a cap 1 kg tighter is refused |


In [1]:
# Cell 1: setup
import hashlib, json, math, os, time
from pathlib import Path
import numpy as np, pandas as pd, requests
from sklearn.ensemble import HistGradientBoostingRegressor

WORK = Path("/content/hp_ai" if Path("/content").exists() else Path.cwd() / "hp_ai")
CACHE = WORK / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
RESULTS = {}
YEAR_START, YEAR_END = pd.Timestamp("2025-01-01"), pd.Timestamp("2026-01-01")
TEST_START = pd.Timestamp("2025-07-01")
ALPHA = 0.01
SESSION = requests.Session()

def record(test_id, status, summary, **metrics):
    RESULTS[test_id] = {"status": status, "summary": summary, "metrics": metrics}
    (WORK / "ai_results.json").write_text(json.dumps(RESULTS, indent=2, default=float))
    print(f"\n{dict(PASS='✅', WARN='⚠️', FAIL='❌', SKIP='⏭️', INFO='ℹ️')[status]} {test_id}: {status} — {summary}")

def cached_get(url, params, tag, pause=1.5):
    key = hashlib.sha256((url + json.dumps(params, sort_keys=True)).encode()).hexdigest()[:20]
    f = CACHE / f"{tag}_{key}.json"
    if f.exists():
        return json.loads(f.read_text())
    wait = 10
    for attempt in range(10):
        time.sleep(pause)
        try:
            r = SESSION.get(url, params=params, timeout=300)
        except requests.RequestException as e:
            print(f"   network {type(e).__name__}; retry in {wait}s"); time.sleep(wait); wait = min(2 * wait, 180); continue
        if r.status_code == 429 or r.status_code >= 500:
            ra = r.headers.get("Retry-After"); w = int(ra) + 1 if ra and ra.isdigit() else wait
            print(f"   HTTP {r.status_code} {tag}; waiting {w}s"); time.sleep(w); wait = min(2 * wait, 180); continue
        if r.status_code >= 400:
            raise RuntimeError(f"{tag}: HTTP {r.status_code}: {r.text[:300]}")
        data = r.json(); f.write_text(json.dumps(data)); return data
    raise RuntimeError(f"{tag}: still failing after 10 attempts; rerun later (cached parts are kept)")

def eds(dataset, filt=None, columns=None, timecol="HourUTC", start=YEAR_START - pd.Timedelta(days=1), end=YEAR_END + pd.Timedelta(days=1)):
    frames = []
    edges = sorted({start, *pd.date_range(start, end, freq="MS"), end})
    for a, b in zip(edges[:-1], edges[1:]):
        p = {"start": a.strftime("%Y-%m-%dT%H:%M"), "end": b.strftime("%Y-%m-%dT%H:%M"), "limit": 0}
        if filt: p["filter"] = json.dumps(filt)
        if columns: p["columns"] = ",".join(columns)
        frames.append(pd.DataFrame(cached_get("https://api.energidataservice.dk/dataset/" + dataset, p, dataset)["records"]))
    df = pd.concat(frames, ignore_index=True)
    df[timecol] = pd.to_datetime(df[timecol])
    lo, hi = df[timecol].min(), df[timecol].max()
    assert lo >= start - pd.Timedelta(days=2) and hi <= end + pd.Timedelta(days=2), f"{dataset}: params ignored? {lo}..{hi}"
    print(f"   {dataset}: {len(df):,} rows")
    return df

print("work dir:", WORK)

work dir: /content/hp_ai


In [2]:
# Cell 2: Energinet data (wind production, consumption-weighted grid CO2 intensity, day-ahead forecasts, buyer load)
st = eds("ProductionConsumptionSettlement", {"PriceArea": ["DK1", "DK2"]})
wcols = [c for c in st.columns if "Wind" in c]
st["wind"] = st[wcols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
st["cons"] = pd.to_numeric(st["GrossConsumptionMWh"], errors="coerce")
wind = st.groupby("HourUTC")["wind"].sum().rename("wind_mwh")
cons_area = st.pivot_table(index="HourUTC", columns="PriceArea", values="cons", aggfunc="sum")

ge = eds("DeclarationGridEmission", {"PriceArea": ["DK1", "DK2"]})
print("   versions:", sorted(ge["Version"].unique()), " methods:", sorted(ge["FuelAllocationMethod"].unique()))
ver = next(v for v in ("Final", "Preliminary", "Initial") if v in set(ge["Version"]))
meth = "125%" if "125%" in set(ge["FuelAllocationMethod"]) else sorted(ge["FuelAllocationMethod"].unique())[0]
g = ge[(ge["Version"] == ver) & (ge["FuelAllocationMethod"] == meth)]
inten_area = g.pivot_table(index="HourUTC", columns="PriceArea", values="CO2PerkWh", aggfunc="mean")
w = cons_area.reindex(inten_area.index)
intensity = ((inten_area * w).sum(axis=1) / w.sum(axis=1)).rename("co2_g_per_kwh")
print(f"   grid intensity: Version={ver}, method={meth} (Danish declaration convention), consumption-weighted DK1+DK2")

fc = eds("Forecasts_Hour", {"PriceArea": ["DK1", "DK2"]})
fc["ForecastDayAhead"] = pd.to_numeric(fc["ForecastDayAhead"], errors="coerce")
fc["key"] = fc["PriceArea"] + "_" + fc["ForecastType"].str.lower().str.replace(" ", "_")
fcp = fc.pivot_table(index="HourUTC", columns="key", values="ForecastDayAhead", aggfunc="sum")
fcp.columns = [f"fc_{c}" for c in fcp.columns]

ld = eds("ConsumptionDK3619IndustryHour", timecol="TimeUTC")
tot = ld.groupby(["DK36Code", "DK36Title"])["Consumption_MWh"].sum().sort_values(ascending=False)
code = next((c for (c, t) in tot.index if any(k in str(t).lower() for k in ("informat", "telekom", "it-"))), tot.index[0][0])
load = ld[ld["DK36Code"] == code].groupby("TimeUTC")["Consumption_MWh"].sum().rename("load_mwh")
load.index.name = "HourUTC"
print("   buyer branch:", code, ld.loc[ld["DK36Code"] == code, "DK36Title"].iloc[0])

E = pd.concat([wind, intensity, load, fcp], axis=1)
E = E[(E.index >= YEAR_START) & (E.index < YEAR_END)].sort_index()
E.to_csv(WORK / "energinet_hourly.csv")
print(E.describe().T[["count", "mean", "min", "max"]].round(1))

   HTTP 429 ProductionConsumptionSettlement; waiting 6s
   HTTP 429 ProductionConsumptionSettlement; waiting 5s
   HTTP 429 ProductionConsumptionSettlement; waiting 5s
   HTTP 429 ProductionConsumptionSettlement; waiting 5s
   ProductionConsumptionSettlement: 17,616 rows
   HTTP 429 DeclarationGridEmission; waiting 60s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   HTTP 429 DeclarationGridEmission; waiting 59s
   DeclarationGridEmission: 52,848 rows
   versions: ['Final', 'Preliminary']  methods: ['1

In [3]:
# Cell 3: day-ahead prices (published ~13:00 CET on D-1, before the 17:30 UTC decision). Elspot hourly to Sep 2025,
# then DayAheadPrices in 15-minute steps (averaged to hours).
sp = eds("Elspotprices", {"PriceArea": ["DK1", "DK2"]}, end=pd.Timestamp("2025-10-01"))
sp["p"] = pd.to_numeric(sp["SpotPriceEUR"], errors="coerce")
p1 = sp.groupby("HourUTC")["p"].mean()
da = eds("DayAheadPrices", {"PriceArea": ["DK1", "DK2"]}, timecol="TimeUTC", start=pd.Timestamp("2025-10-01"))
da["p"] = pd.to_numeric(da["DayAheadPriceEUR"], errors="coerce")
p2 = da.groupby(da["TimeUTC"].dt.floor("h"))["p"].mean()
price = pd.concat([p1[p1.index < pd.Timestamp("2025-10-01")], p2]).sort_index()
price = price[~price.index.duplicated()].rename("price_eur")
E["price_eur"] = price.reindex(E.index)
print(f"   prices: {E['price_eur'].notna().mean():.1%} of 2025 hours")

   HTTP 429 Elspotprices; waiting 300s
   HTTP 429 Elspotprices; waiting 298s
   HTTP 429 Elspotprices; waiting 300s
   HTTP 429 Elspotprices; waiting 300s
   HTTP 429 Elspotprices; waiting 299s
   HTTP 429 Elspotprices; waiting 299s
   HTTP 429 Elspotprices; waiting 300s
   HTTP 429 Elspotprices; waiting 300s
   HTTP 429 Elspotprices; waiting 300s
   Elspotprices: 13,150 rows
   HTTP 429 DayAheadPrices; waiting 5s
   DayAheadPrices: 17,864 rows
   prices: 100.0% of 2025 hours


In [4]:
import requests

url = "https://api.energidataservice.dk/dataset/ProductionConsumptionSettlement"
r = requests.get(
    url,
    params={"limit": 1, "filter": '{"PriceArea":["DK1","DK2"]}'},
    timeout=30
)
print("Status:", r.status_code)
print(r.text[:500])

Status: 200
{"total":1,"filters":"{\"PriceArea\":[\"DK1\",\"DK2\"]}","limit":1,"dataset":"ProductionConsumptionSettlement","records":[{"HourUTC":"2026-10-05T23:00:00","HourDK":"2026-10-06T01:00:00","PriceArea":"DK1","CentralPowerMWh":598.834781,"LocalPowerMWh":153.187121,"CommercialPowerMWh":58.650143,"LocalPowerSelfConMWh":null,"OffshoreWindLt100MW_MWh":31.650670,"OffshoreWindGe100MW_MWh":1110.705596,"OnshoreWindLt50kW_MWh":5.013001,"OnshoreWindGe50kW_MWh":1274.975725,"HydroPowerMWh":1.224135,"SolarPowerLt


In [5]:
# Cell 4: K1, day-ahead CO2-intensity forecast. Feature set chosen on VALIDATION (Apr-Jun), TEST (Jul-Dec) run once.
VAL_START, TEST_START = pd.Timestamp("2025-04-01"), pd.Timestamp("2025-07-01")
DK_HOLIDAYS = {"2025-01-01", "2025-04-17", "2025-04-18", "2025-04-20", "2025-04-21", "2025-05-16", "2025-05-29",
               "2025-06-05", "2025-06-08", "2025-06-09", "2025-12-24", "2025-12-25", "2025-12-26", "2025-12-31"}
I = E["co2_g_per_kwh"]
fc_cols = [c for c in E.columns if c.startswith("fc_")]

def gbr():
    return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0, random_state=0)

def co2_features(idx, use_price):
    F = pd.DataFrame(index=idx)
    for c in fc_cols:
        F[c] = E[c].reindex(idx).to_numpy()
    F["fc_wind_total"] = F[[c for c in fc_cols if "wind" in c]].sum(axis=1, min_count=1)
    F["hour"], F["dow"], F["month"] = idx.hour, idx.dayofweek, idx.month
    F["holiday"] = [d in DK_HOLIDAYS for d in idx.strftime("%Y-%m-%d")]
    for lag in (48, 72, 168):
        F[f"lag{lag}"] = I.reindex(idx - pd.Timedelta(hours=lag)).to_numpy()
    F["lag_mean_week"] = pd.DataFrame({k: I.reindex(idx - pd.Timedelta(hours=24 * k)).to_numpy() for k in range(2, 9)}, index=idx).mean(axis=1)
    if use_price:
        pr = E["price_eur"].reindex(idx)
        F["price"] = pr.to_numpy()
        F["price_rank_day"] = pr.groupby(idx.normalize()).rank(pct=True).to_numpy()
        F["price_minus_daymean"] = (pr - pr.groupby(idx.normalize()).transform("mean")).to_numpy()
    return F

def walk_forward(start, end, use_price):
    out = pd.Series(dtype=float)
    for M in pd.date_range(start, end - pd.Timedelta(days=1), freq="MS"):
        tr_idx = I.dropna().index[I.dropna().index < M - pd.Timedelta(hours=6.5)]
        X = co2_features(tr_idx, use_price); ok = X.notna().sum(axis=1) > 3
        m = gbr().fit(X[ok], I.loc[tr_idx][ok])
        te = E.index[(E.index >= M) & (E.index < min(M + pd.offsets.MonthBegin(1), end))]
        out = pd.concat([out, pd.Series(m.predict(co2_features(te, use_price)), index=te)])
    return out

def mae(pred, start, end):
    a = I.loc[start:end - pd.Timedelta(hours=1)]
    return float((pred.reindex(a.index) - a).abs().mean())

val = {up: mae(walk_forward(VAL_START, TEST_START, up), VAL_START, TEST_START) for up in (False, True)}
USE_PRICE = val[True] < val[False]
print(f"   VALIDATION MAE: without prices {val[False]:.1f}, with prices {val[True]:.1f} g/kWh -> frozen: use_price={USE_PRICE}")
pred_I = walk_forward(TEST_START, YEAR_END, USE_PRICE)          # TEST, run once
T = pd.DataFrame({"actual": I, "model": pred_I, "lag48": I.shift(48), "lag168": I.shift(168)}).loc[TEST_START:].dropna()
m_mae = {k: float((T[k] - T["actual"]).abs().mean()) for k in ("model", "lag48", "lag168")}
base = min(m_mae["lag48"], m_mae["lag168"])
record("K1_co2_forecast", "PASS" if m_mae["model"] <= 0.7 * base else "FAIL",
       f"TEST MAE {m_mae['model']:.1f} vs best persistence {base:.1f} g/kWh ({1 - m_mae['model'] / base:.0%} better; bar 30%)",
       validation=val, use_price=USE_PRICE, **m_mae)

/tmp/ipykernel_1874/2148163767.py:35: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  out = pd.concat([out, pd.Series(m.predict(co2_features(te, use_price)), index=te)])
/tmp/ipykernel_1874/2148163767.py:35: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  out = pd.concat([out, pd.Series(m.predict(co2_features(te, use_price)), index=te)])


   VALIDATION MAE: without prices 23.8, with prices 21.7 g/kWh -> frozen: use_price=True


/tmp/ipykernel_1874/2148163767.py:35: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  out = pd.concat([out, pd.Series(m.predict(co2_features(te, use_price)), index=te)])



✅ K1_co2_forecast: PASS — TEST MAE 20.3 vs best persistence 38.4 g/kWh (47% better; bar 30%)


In [6]:
# Cell 5: K2/K3, carbon-aware shifting of the buyer's flexible load (same scenario as HourProof: 10 % of each hour's load
# may move within the same UTC day, no hour above +25 %). Decisions use only the forecast; outcomes use actual data.
FLEX, CAP = 0.10, 0.25
D = E.loc[TEST_START:].copy()
k = E["load_mwh"].sum() / E["wind_mwh"].sum()
D["base"] = k * D["wind_mwh"]
D["load_hat"] = E["load_mwh"].shift(168).reindex(D.index)          # last week: the best load forecast we have
D["I_model"] = pred_I.reindex(D.index)
D["I_persist"] = I.shift(168).reindex(D.index)
D["I_perfect"] = D["co2_g_per_kwh"]
D = D.dropna(subset=["load_mwh", "load_hat", "I_model", "I_persist", "co2_g_per_kwh", "base"])
D = D[D.groupby(D.index.normalize())["load_mwh"].transform("size") == 24]

def plan_shift(load_hat, prio):
    """Move flexible load from the highest-priority (dirtiest) hours to the lowest; returns planned shift per hour."""
    n = len(load_hat); shift = np.zeros(n)
    give = FLEX * load_hat.copy(); room = CAP * load_hat.copy()
    donors = list(np.argsort(-prio)); receivers = list(np.argsort(prio))
    for d in donors:
        for r in receivers:
            if give[d] <= 1e-9: break
            if prio[r] >= prio[d] or room[r] <= 1e-9 or r == d: continue
            m = min(give[d], room[r]); shift[d] -= m; shift[r] += m; give[d] -= m; room[r] -= m
    return shift

def execute(load, shift):
    out = np.minimum(-np.minimum(shift, 0), FLEX * load); moved = out.sum()
    inc = np.maximum(shift, 0); add = inc / inc.sum() * moved if inc.sum() > 0 else np.zeros_like(load)
    return load - out + add

def run(prio_col):
    new = pd.Series(index=D.index, dtype=float)
    for day, g in D.groupby(D.index.normalize()):
        if prio_col is None:
            new.loc[g.index] = g["load_mwh"].to_numpy(); continue
        prio = g[prio_col].to_numpy() if prio_col != "mwh" else -(k * g["fc_wind_total" if "fc_wind_total" in g else "wind_mwh"].fillna(0).to_numpy() - g["load_hat"].to_numpy())
        new.loc[g.index] = execute(g["load_mwh"].to_numpy(), plan_shift(g["load_hat"].to_numpy(), prio))
    emis_t = float((new * D["co2_g_per_kwh"]).sum() / 1000)
    resid_t = float(((new - np.minimum(new, D["base"])) * D["co2_g_per_kwh"]).sum() / 1000)
    cov = float(np.minimum(new, D["base"]).sum() / new.sum())
    return {"emissions_t": emis_t, "residual_unmatched_t": resid_t, "coverage": cov}

D["fc_wind_total"] = D[[c for c in fc_cols if "wind" in c]].sum(axis=1, min_count=1)
R = {"no_shift": run(None), "shift_for_mwh": run("mwh"), "carbon_persist": run("I_persist"),
     "carbon_model": run("I_model"), "carbon_perfect": run("I_perfect")}
print(pd.DataFrame(R).T.round(4).to_string())
red = {k_: 1 - R[k_]["emissions_t"] / R["no_shift"]["emissions_t"] for k_ in ("carbon_persist", "carbon_model", "carbon_perfect", "shift_for_mwh")}
print("   emission reduction vs no shifting:", {k_: f"{v:.2%}" for k_, v in red.items()})
cov_cost = 100 * (R["shift_for_mwh"]["coverage"] - R["carbon_model"]["coverage"])
record("K2_carbon_shifting", "PASS" if red["carbon_model"] >= 0.01 and cov_cost <= 0.5 else ("WARN" if red["carbon_model"] > 0 else "FAIL"),
       f"learned-forecast shifting cuts the buyer's location-based emissions {red['carbon_model']:.2%} "
       f"({R['no_shift']['emissions_t'] - R['carbon_model']['emissions_t']:,.0f} t in 6 months); coverage vs MWh-shifting {-cov_cost:+.2f} pp",
       results=R, reductions=red)
ok3 = red["carbon_model"] >= 0.5 * red["carbon_perfect"] and red["carbon_model"] >= 1.25 * max(red["carbon_persist"], 1e-9)
record("K3_ai_is_load_bearing", "PASS" if ok3 else "FAIL",
       f"learned forecast reaches {red['carbon_model'] / red['carbon_perfect']:.0%} of the perfect-forecast reduction; "
       f"persistence reaches {red['carbon_persist'] / red['carbon_perfect']:.0%}", **{k_: float(v) for k_, v in red.items()})

                emissions_t  residual_unmatched_t  coverage
no_shift         48840.1870            21837.2252    0.7045
shift_for_mwh    47889.0786            20776.8662    0.7147
carbon_persist   48546.4960            21737.7775    0.7003
carbon_model     47631.7210            20592.3076    0.7136
carbon_perfect   46856.4615            20071.8500    0.7137
   emission reduction vs no shifting: {'carbon_persist': '0.60%', 'carbon_model': '2.47%', 'carbon_perfect': '4.06%', 'shift_for_mwh': '1.95%'}

✅ K2_carbon_shifting: PASS — learned-forecast shifting cuts the buyer's location-based emissions 2.47% (1,208 t in 6 months); coverage vs MWh-shifting -0.11 pp

✅ K3_ai_is_load_bearing: PASS — learned forecast reaches 61% of the perfect-forecast reduction; persistence reaches 15%


In [7]:
# Cell 6: K4, zero-knowledge proof of hourly residual emissions (load private, intensities public)
import subprocess, tarfile, urllib.request
BIN = WORK / "bin"; BIN.mkdir(exist_ok=True); os.environ["PATH"] = f"{BIN}:{os.environ['PATH']}"
def fetch_tar(url):
    f = WORK / "t.tgz"; urllib.request.urlretrieve(url, f)
    with tarfile.open(f) as t: t.extractall(BIN)
if not (BIN / "nargo").exists():
    fetch_tar("https://github.com/noir-lang/noir/releases/download/v1.0.0-beta.22/nargo-x86_64-unknown-linux-gnu.tar.gz")
if not (BIN / "bb").exists():
    fetch_tar("https://github.com/AztecProtocol/barretenberg/releases/download/v5.0.0-nightly.20260522/barretenberg-amd64-linux.tar.gz")
for b in BIN.iterdir(): b.chmod(0o755)
C = WORK / "emissions"; (C / "src").mkdir(parents=True, exist_ok=True)
(C / "Nargo.toml").write_text('[package]\nname = "emissions"\ntype = "bin"\nauthors = [""]\n\n[dependencies]\n')
(C / "src" / "main.nr").write_text('''
use std::hash::pedersen_hash;
global N: u32 = 168;
// private: hourly load (kWh), salt.  public: certificates (kWh), grid intensity (g/kWh), coverage bar, emissions cap (kg)
// proves: coverage >= threshold AND unmatched (residual) emissions <= cap, and outputs the load commitment
fn main(load: [u32; N], salt: Field, threshold_bps: pub u32, cap_kg: pub u64, cert: pub [u32; N], intensity: pub [u32; N]) -> pub Field {
    let mut matched: u64 = 0; let mut total: u64 = 0; let mut resid_g: u64 = 0;
    for i in 0..N {
        let l = load[i] as u64; let c = cert[i] as u64;
        let m = if c < l { c } else { l };
        matched += m; total += l;
        resid_g += (l - m) * (intensity[i] as u64);
    }
    assert(total > 0);
    assert(matched * 10000 >= (threshold_bps as u64) * total);
    assert(resid_g <= cap_kg * 1000);
    let mut buf: [Field; N + 1] = [0; N + 1];
    for i in 0..N { buf[i] = load[i] as Field; }
    buf[N] = salt;
    pedersen_hash(buf)
}
''')
w0 = D.index.normalize().unique()[7 * 6]          # a real week inside the test period
wk = D.loc[w0: w0 + pd.Timedelta(hours=167)]
if len(wk) == 168:
    load_kwh = (wk["load_mwh"] * 1000).round().astype(int).tolist()
    cert_kwh = (np.floor(wk["base"]).astype(int) * 1000).tolist()
    inten = wk["co2_g_per_kwh"].round().astype(int).clip(lower=0).tolist()
    matched = sum(min(a, b) for a, b in zip(load_kwh, cert_kwh)); tot = sum(load_kwh)
    resid_g = sum((a - min(a, b)) * i for a, b, i in zip(load_kwh, cert_kwh, inten))
    bps = matched * 10000 // tot; cap = -(-resid_g // 1000)
    toml = (f"load = {load_kwh}\nsalt = \"12345\"\nthreshold_bps = {bps}\ncap_kg = {cap}\ncert = {cert_kwh}\nintensity = {inten}\n")
    (C / "Prover.toml").write_text(toml)
    def sh(cmd):
        p = subprocess.run(cmd, shell=True, cwd=C, capture_output=True, text=True); return p.returncode, p.stdout + p.stderr
    sh("nargo compile"); _, gates = sh("bb gates -b target/emissions.json")
    rc_ok, _ = sh("nargo execute")
    (C / "Prover.toml").write_text(toml.replace(f"cap_kg = {cap}", f"cap_kg = {cap - 1}"))
    rc_under, _ = sh("nargo execute")
    (C / "Prover.toml").write_text(toml); sh("nargo execute")
    t0 = time.time(); rc_p, outp = sh("bb prove -b target/emissions.json -w target/emissions.gz -o target -t evm --write_vk"); pt = time.time() - t0
    rc_v, _ = sh("bb verify -k target/vk -p target/proof -i target/public_inputs -t evm")
    sh("bb write_solidity_verifier -k target/vk -o target/Verifier.sol -t evm")
    size_note = (C / "target" / "Verifier.sol").stat().st_size if (C / "target" / "Verifier.sol").exists() else None
    import re as _re
    cs = _re.search(r'"circuit_size":\s*(\d+)', gates)
    ok4 = rc_ok == 0 and rc_under != 0 and rc_p == 0 and rc_v == 0 and pt <= 60
    record("K4_zk_emissions", "PASS" if ok4 else "FAIL",
           f"week {w0:%Y-%m-%d}: coverage >= {bps / 100:.2f}% and residual <= {cap / 1000:,.1f} t; prove {'OK' if rc_p == 0 else 'FAILED: ' + outp.strip().splitlines()[-1][:120]} ({pt:.1f}s), verify {'OK' if rc_v == 0 else 'FAILED'}; "
           f"cap 1 kg lower refused: {rc_under != 0}; circuit {cs.group(1) if cs else '?'} gates",
           prove_s=pt, honest_ok=rc_ok == 0, tighter_cap_refused=rc_under != 0, native_verify=rc_v == 0,
           solidity_source_bytes=size_note, circuit_size=int(cs.group(1)) if cs else None)
else:
    record("K4_zk_emissions", "SKIP", "no complete test week found")

/tmp/ipykernel_1874/723837964.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(f) as t: t.extractall(BIN)



✅ K4_zk_emissions: PASS — week 2025-08-12: coverage >= 44.76% and residual <= 1,842.7 t; prove OK (6.5s), verify OK; cap 1 kg lower refused: True; circuit 62240 gates


In [8]:
# Cell 7: verdict
print(f"{'test':<24}{'status':<7}summary")
for k_, v in RESULTS.items():
    print(f"{k_:<24}{v['status']:<7}{v['summary']}")
go = all(RESULTS.get(t, {}).get("status") == "PASS" for t in ("K1_co2_forecast", "K2_carbon_shifting", "K3_ai_is_load_bearing", "K4_zk_emissions"))
print("\nCARBON-AWARE HOURPROOF:", "BUILD" if go else "DO NOT BUILD — see the lines above")
print("Caveats: average (location-based) emission factors, not marginal; shifting share 10 % is a declared scenario "
      "parameter; one test half-year; Energinet's Final declaration data is used for training as if available in time.")
print(json.dumps(RESULTS, indent=1, default=float)[:6000])

test                    status summary
K1_co2_forecast         PASS   TEST MAE 20.3 vs best persistence 38.4 g/kWh (47% better; bar 30%)
K2_carbon_shifting      PASS   learned-forecast shifting cuts the buyer's location-based emissions 2.47% (1,208 t in 6 months); coverage vs MWh-shifting -0.11 pp
K3_ai_is_load_bearing   PASS   learned forecast reaches 61% of the perfect-forecast reduction; persistence reaches 15%
K4_zk_emissions         PASS   week 2025-08-12: coverage >= 44.76% and residual <= 1,842.7 t; prove OK (6.5s), verify OK; cap 1 kg lower refused: True; circuit 62240 gates

CARBON-AWARE HOURPROOF: BUILD
Caveats: average (location-based) emission factors, not marginal; shifting share 10 % is a declared scenario parameter; one test half-year; Energinet's Final declaration data is used for training as if available in time.
{
 "K1_co2_forecast": {
  "status": "PASS",
  "summary": "TEST MAE 20.3 vs best persistence 38.4 g/kWh (47% better; bar 30%)",
  "metrics": {
   "validation":